In [1]:
# 让 notebook 无论从哪个目录启动都能 import retailinsight
import sys
from pathlib import Path

_candidates = [Path.cwd(), *Path.cwd().parents]
_root = next(p for p in _candidates if (p / "retailinsight" / "__init__.py").exists())
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
print("项目根目录:", _root)


项目根目录: /Users/lizhipeng/个人成长/职业发展/求职面试/项目/RetailInsight


# 04 · 关联规则挖掘
> 目标：把「一起买」的商品组合变成可执行的陈列与捆绑策略。

In [2]:
import pandas as pd
from retailinsight.io.loader import read_parquet
from retailinsight.settings import MART_RULES

rules = read_parquet(MART_RULES)
print(f"共 {len(rules)} 条规则（min_support 1% / min_confidence 30% / lift > 1.5）")
rules.head(10)


共 177 条规则（min_support 1% / min_confidence 30% / lift > 1.5）


,antecedents,consequents,support,confidence,lift
0,护发素,洗发水,0.0491,0.6439,8.3033
1,洗发水,护发素,0.0491,0.6331,8.3033
2,护发素/粮油米面,洗发水,0.0213,0.5980,7.7107
3,洗发水/粮油米面,护发素,0.0213,0.5879,7.7102
4,洗发水/调味品,护发素,0.0153,0.5661,7.4241
5,护发素/调味品,洗发水,0.0153,0.5740,7.4016
6,护发素/饮料,洗发水,0.0177,0.5647,7.2821
7,洗发水/饮料,护发素,0.0177,0.5516,7.2338
8,护发素/面包糕点,洗发水,0.0118,0.5122,6.6050
9,洗发水/面包糕点,护发素,0.0118,0.5010,6.5709


## 4.1 规则强度分布

In [3]:
print("lift 分布:", rules["lift"].describe().round(2).to_dict())
print("\nconfidence 分布:", rules["confidence"].describe().round(2).to_dict())
rules["lift_bin"] = pd.cut(rules["lift"], [1.5, 2, 3, 5, 10], right=False)
rules.groupby("lift_bin", observed=True)["lift"].size()


lift 分布: {'count': 177.0, 'mean': 2.29, 'std': 1.49, 'min': 1.5, '25%': 1.58, '50%': 1.68, '75%': 2.11, 'max': 8.3}

confidence 分布: {'count': 177.0, 'mean': 0.5, 'std': 0.13, 'min': 0.3, '25%': 0.35, '50%': 0.53, '75%': 0.57, 'max': 0.79}


lift_bin
[1.5, 2.0)     125
[2.0, 3.0)      26
[3.0, 5.0)      13
[5.0, 10.0)     13
Name: lift, dtype: int64

## 4.2 Top 20 强关联规则

In [4]:
rules.nlargest(20, "lift")[["antecedents", "consequents", "support",
                                  "confidence", "lift"]].round(4)


,antecedents,consequents,support,confidence,lift
0,护发素,洗发水,0.0491,0.6439,8.3033
1,洗发水,护发素,0.0491,0.6331,8.3033
2,护发素/粮油米面,洗发水,0.0213,0.5980,7.7107
3,洗发水/粮油米面,护发素,0.0213,0.5879,7.7102
4,洗发水/调味品,护发素,0.0153,0.5661,7.4241
5,护发素/调味品,洗发水,0.0153,0.5740,7.4016
6,护发素/饮料,洗发水,0.0177,0.5647,7.2821
7,洗发水/饮料,护发素,0.0177,0.5516,7.2338
8,护发素/面包糕点,洗发水,0.0118,0.5122,6.6050
9,洗发水/面包糕点,护发素,0.0118,0.5010,6.5709


## 4.3 埋点回收验证：4 组预埋组合

In [5]:
from retailinsight.io.schema import ANCHOR_COMBOS

for a, b, _target in ANCHOR_COMBOS:
    hit = rules[(rules["antecedents"].astype(str).str.contains(a, regex=False))
                & (rules["consequents"].astype(str).str.contains(b, regex=False))]
    print(f"  {a} → {b}: {'✅ lift=' + format(hit['lift'].max(), '.2f') if len(hit) else '❌ 未回收'}")


  啤酒 → 纸尿裤: ❌ 未回收
  火锅底料 → 牛羊肉: ✅ lift=5.02
  面包糕点 → 乳制品: ✅ lift=2.29
  洗发水 → 护发素: ✅ lift=8.30


## 4.4 高频共同出现品类对

In [6]:
# 前后件都可能是多品类组合，逐行展开成「品类对」
_rows = []
for _, r in rules.iterrows():
    for a in str(r["antecedents"]).split("/"):
        for b in str(r["consequents"]).split("/"):
            if a != b:
                _rows.append({"a": a, "b": b, "lift": float(r["lift"]),
                              "confidence": float(r["confidence"])})
_pair_df = pd.DataFrame(_rows)

pairs = (_pair_df.groupby(["a", "b"])
         .agg(n=("lift", "size"), lift=("lift", "max"), confidence=("confidence", "max"))
         .sort_values(["n", "lift"], ascending=False))
pairs.head(15)


n    lift  confidence
a    b                           
面包糕点 乳制品   49  2.2884      0.5905
饮料   乳制品   21  2.0916      0.5905
乳制品  面包糕点  19  2.2352      0.6297
进口食品 面包糕点  18  2.1802      0.6297
     乳制品   16  2.2521      0.5905
冷冻食品 粮油米面  13  2.1906      0.7693
粮油米面 乳制品   13  1.9659      0.5821
咖啡茶饮 乳制品   12  2.2884      0.5872
饮料   面包糕点  12  2.1906      0.6297
调味品  粮油米面  11  5.0169      0.7883
咖啡茶饮 面包糕点  11  2.2352      0.6269
冷冻食品 乳制品   11  2.0916      0.5821
纸尿裤  啤酒    10  3.6669      0.7672
乳制品  饮料    10  2.2352      0.3935
火锅底料 牛羊肉    9  5.0169      0.5933